# Q5: Language Modeling

This notebook investigates language modeling using two approaches on the **WikiText-2** benchmark:

| # | Model | Paradigm |
|---|---|---|
| 1 | LSTM Language Model | Trained from scratch on a WikiText-2 subset |
| 2 | GPT-2 (`gpt2`) | Pre-trained Transformer decoder, evaluated without fine-tuning |

Both models are evaluated with **perplexity** (the standard intrinsic metric for language models) on the same held-out test subset, and are used to generate short text continuations for qualitative comparison.

## 1. Environment Setup

Required libraries: `datasets` (WikiText-2 loading), `transformers` (GPT-2 model and tokenizer), and `accelerate` (efficient inference). Core scientific stack (`numpy`, `pandas`, `torch`) handles numerics, batching, and training. A global seed of **42** is applied across Python, NumPy, and PyTorch.

In [1]:
!pip -q install datasets transformers accelerate

In [2]:
import os
import re
import math
import time
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Imports completed successfully.")
print("Random seed:", SEED)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.")

Imports completed successfully.
Random seed: 42
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Dataset: WikiText-2

**WikiText-2** (Merity et al., 2016) is a standard language modeling benchmark derived from verified Good and Featured Wikipedia articles. It preserves punctuation, capitalization, and document structure, making it more realistic than the older Penn Treebank benchmark.

Key properties: ~2M training tokens, ~217k validation tokens, ~245k test tokens. Empty lines and section headers are filtered out before use. To keep training time manageable, a subset of **8,000 training lines**, **1,000 validation lines**, and **1,000 test lines** is selected with a fixed random seed.

In [3]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

print(dataset)

print("\nColumn names:")
for split in dataset.keys():
    print(split, dataset[split].column_names)

print("\nFirst non-empty training examples:")
count = 0
for example in dataset["train"]:
    text = example["text"].strip()
    if text:
        print("-" * 80)
        print(text[:500])
        count += 1
    if count == 3:
        break

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Column names:
test ['text']
train ['text']
validation ['text']

First non-empty training examples:
--------------------------------------------------------------------------------
= Valkyria Chronicles III =
--------------------------------------------------------------------------------
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors ,

In [4]:
# Basic dataset statistics


def non_empty_texts(split):
    return [ex["text"].strip() for ex in dataset[split] if ex["text"].strip()]

train_texts_full = non_empty_texts("train")
valid_texts_full = non_empty_texts("validation")
test_texts_full = non_empty_texts("test")

print("Non-empty documents/lines:")
print("Train:", len(train_texts_full))
print("Validation:", len(valid_texts_full))
print("Test:", len(test_texts_full))

def simple_lm_tokenize(text):
    text = text.lower().strip()
    text = re.sub(r"([.!?,;:()\[\]\"'])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

train_token_counts = [len(simple_lm_tokenize(text)) for text in train_texts_full]
valid_token_counts = [len(simple_lm_tokenize(text)) for text in valid_texts_full]
test_token_counts = [len(simple_lm_tokenize(text)) for text in test_texts_full]

stats_df = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "non_empty_lines": [len(train_texts_full), len(valid_texts_full), len(test_texts_full)],
    "mean_tokens_per_line": [
        np.mean(train_token_counts),
        np.mean(valid_token_counts),
        np.mean(test_token_counts)
    ],
    "median_tokens_per_line": [
        np.median(train_token_counts),
        np.median(valid_token_counts),
        np.median(test_token_counts)
    ],
    "max_tokens_per_line": [
        np.max(train_token_counts),
        np.max(valid_token_counts),
        np.max(test_token_counts)
    ],
})

stats_df

Non-empty documents/lines:
Train: 23767
Validation: 2461
Test: 2891


,split,non_empty_lines,mean_tokens_per_line,median_tokens_per_line,max_tokens_per_line
0,train,23767,87.853074,76.0,721
1,validation,2461,88.524584,76.0,434
2,test,2891,84.874092,68.0,483


## 3. Preprocessing and Vocabulary Construction

Text is lowercased and tokenized with a regex-based word-level tokenizer. A vocabulary is built exclusively from the training subset using a minimum frequency threshold of **3** — tokens appearing fewer than 3 times are discarded and mapped to `<unk>` at runtime. Three special tokens are reserved: `<pad>` (sequence padding), `<unk>` (unknown words), and `<eos>` (appended at the end of each line to mark sentence boundaries).

In [5]:
TRAIN_LINES = 8000
VALID_LINES = 1000
TEST_LINES = 1000

random.Random(SEED).shuffle(train_texts_full)
random.Random(SEED).shuffle(valid_texts_full)
random.Random(SEED).shuffle(test_texts_full)

train_texts = train_texts_full[:TRAIN_LINES]
valid_texts = valid_texts_full[:VALID_LINES]
test_texts = test_texts_full[:TEST_LINES]

print("Train lines:", len(train_texts))
print("Validation lines:", len(valid_texts))
print("Test lines:", len(test_texts))

print("\nSample training line:")
print(train_texts[0][:500])

Train lines: 8000
Validation lines: 1000
Test lines: 1000

Sample training line:
When Wheeler was four , his father was appointed chief leader writer for the Bradford Observer . The family relocated to Saltaire , a village northwest of Bradford , a cosmopolitan city in Yorkshire , northeast England , which was then in the midst of the wool trade boom . Wheeler was inspired by the moors surrounding Saltaire and fascinated by the area 's archaeology . He later wrote about discovering a late prehistoric cup @-@ marked stone , searching for lithics on Ilkley Moor , and digging i


In [6]:
# Build word-level vocabulary

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
EOS_TOKEN = "<eos>"

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, EOS_TOKEN]

def build_lm_vocab(texts, min_freq=3, max_vocab_size=20000):
    counter = Counter()

    for text in texts:
        tokens = simple_lm_tokenize(text)
        counter.update(tokens)
        counter.update([EOS_TOKEN])

    vocab = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}

    for token, freq in counter.most_common():
        if freq < min_freq:
            continue
        if token in vocab:
            continue
        if len(vocab) >= max_vocab_size:
            break
        vocab[token] = len(vocab)

    return vocab, counter

lm_vocab, lm_counter = build_lm_vocab(
    train_texts,
    min_freq=3,
    max_vocab_size=20000
)

lm_itos = {idx: token for token, idx in lm_vocab.items()}

PAD_IDX = lm_vocab[PAD_TOKEN]
UNK_IDX = lm_vocab[UNK_TOKEN]
EOS_IDX = lm_vocab[EOS_TOKEN]

print("Vocabulary size:", len(lm_vocab))
print("Most common tokens:")
print(lm_counter.most_common(30))

Vocabulary size: 16174
Most common tokens:
[('the', 44470), (',', 34759), ('.', 28776), ('of', 19194), ('and', 17052), ('in', 15389), ('to', 13136), ('a', 12198), ('=', 9886), ('"', 9307), ('<eos>', 8000), ('was', 7120), ("'", 6372), ('@-@', 5696), ('s', 5119), ('on', 5045), ('as', 5013), ('that', 4827), ('for', 4687), ('with', 4472), ('@', 4340), ('by', 4300), ('(', 4089), (')', 4087), ('is', 3845), ('it', 3185), ('from', 3182), ('his', 3027), ('at', 3016), ('he', 2927)]


In [7]:
# Encoding helper functions

def encode_lm_text(text, vocab):
    tokens = simple_lm_tokenize(text)
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokens]
    ids.append(vocab[EOS_TOKEN])
    return ids

def decode_lm_ids(ids, itos):
    tokens = []
    for idx in ids:
        token = itos.get(int(idx), UNK_TOKEN)
        if token == EOS_TOKEN:
            break
        if token != PAD_TOKEN:
            tokens.append(token)
    return " ".join(tokens)

sample_text = train_texts[0]
encoded_sample = encode_lm_text(sample_text, lm_vocab)

print("Original sample:")
print(sample_text[:500])

print("\nEncoded sample first 50 IDs:")
print(encoded_sample[:50])

print("\nDecoded sample first 50 tokens:")
print(decode_lm_ids(encoded_sample[:50], lm_itos))

Original sample:
When Wheeler was four , his father was appointed chief leader writer for the Bradford Observer . The family relocated to Saltaire , a village northwest of Bradford , a cosmopolitan city in Yorkshire , northeast England , which was then in the midst of the wool trade boom . Wheeler was inspired by the moors surrounding Saltaire and fascinated by the area 's archaeology . He later wrote about discovering a late prehistoric cup @-@ marked stone , searching for lithics on Ilkley Moor , and digging i

Encoded sample first 50 IDs:
[59, 1424, 13, 137, 4, 29, 492, 13, 745, 1079, 1713, 668, 20, 3, 3345, 5278, 5, 3, 214, 7665, 9, 1, 4, 10, 790, 1322, 6, 3345, 4, 10, 9508, 89, 8, 5279, 4, 1665, 233, 4, 35, 13, 117, 8, 3, 7666, 6, 3, 1, 986, 3491, 5]

Decoded sample first 50 tokens:
when wheeler was four , his father was appointed chief leader writer for the bradford observer . the family relocated to <unk> , a village northwest of bradford , a cosmopolitan city in yorkshire , nor

## 4. Language Modeling Sequences

All training text is concatenated into a single continuous token stream and then sliced into non-overlapping fixed-length blocks of **64 tokens**. For each block, the input is tokens `[0 … 63]` and the target is tokens `[1 … 64]` — a one-position right shift. This standard language modeling objective trains the model to predict the next token at every position simultaneously (teacher-forced cross-entropy over the full sequence).

In [8]:
# Convert text into continuous token streams


def texts_to_token_stream(texts, vocab):
    token_ids = []

    for text in texts:
        token_ids.extend(encode_lm_text(text, vocab))

    return token_ids

train_token_stream = texts_to_token_stream(train_texts, lm_vocab)
valid_token_stream = texts_to_token_stream(valid_texts, lm_vocab)
test_token_stream = texts_to_token_stream(test_texts, lm_vocab)

print("Train token stream length:", len(train_token_stream))
print("Validation token stream length:", len(valid_token_stream))
print("Test token stream length:", len(test_token_stream))

Train token stream length: 714055
Validation token stream length: 90848
Test token stream length: 87346


In [9]:
# PyTorch dataset for language modeling


class LanguageModelingDataset(Dataset):
    def __init__(self, token_stream, seq_len):
        self.token_stream = token_stream
        self.seq_len = seq_len

    def __len__(self):
        return (len(self.token_stream) - 1) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len

        x = torch.tensor(
            self.token_stream[start:end],
            dtype=torch.long
        )

        y = torch.tensor(
            self.token_stream[start + 1:end + 1],
            dtype=torch.long
        )

        return x, y


SEQ_LEN = 50
BATCH_SIZE = 64

train_lm_dataset = LanguageModelingDataset(train_token_stream, SEQ_LEN)
valid_lm_dataset = LanguageModelingDataset(valid_token_stream, SEQ_LEN)
test_lm_dataset = LanguageModelingDataset(test_token_stream, SEQ_LEN)

train_lm_loader = DataLoader(
    train_lm_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_lm_loader = DataLoader(
    valid_lm_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_lm_loader = DataLoader(
    test_lm_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

x_batch, y_batch = next(iter(train_lm_loader))

print("Number of train sequences:", len(train_lm_dataset))
print("Number of validation sequences:", len(valid_lm_dataset))
print("Number of test sequences:", len(test_lm_dataset))

print("\nInput batch shape:", x_batch.shape)
print("Target batch shape:", y_batch.shape)

print("\nDecoded input example:")
print(decode_lm_ids(x_batch[0].tolist(), lm_itos))

print("\nDecoded target example:")
print(decode_lm_ids(y_batch[0].tolist(), lm_itos))

Number of train sequences: 14281
Number of validation sequences: 1816
Number of test sequences: 1746

Input batch shape: torch.Size([64, 50])
Target batch shape: torch.Size([64, 50])

Decoded input example:
smashed the <unk> . she threw her hands up in shock , but was not injured . later , while sitting with the foreign minister , protesters threw tomatoes at her . the tomatoes hit the foreign minister and <unk> on eva ' s dress . after these two events

Decoded target example:
the <unk> . she threw her hands up in shock , but was not injured . later , while sitting with the foreign minister , protesters threw tomatoes at her . the tomatoes hit the foreign minister and <unk> on eva ' s dress . after these two events ,


## 5. LSTM Language Model

The LSTM language model consists of: an **embedding layer** that maps each token ID to a dense vector, a **multi-layer LSTM** that processes the sequence and maintains a hidden state across time steps, a **dropout layer** applied between layers and on the output, and a **linear projection** that maps the hidden state to logits over the full vocabulary at each position.

The first language model is an LSTM trained from scratch on the selected WikiText-2 training subset. The model receives a sequence of tokens and predicts the next token at every position. Cross-entropy loss is used for training, and perplexity is computed as the exponential of the average loss.

In [10]:
# Define LSTM language model

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x: [batch_size, seq_len]

        embedded = self.dropout(self.embedding(x))
        # embedded: [batch_size, seq_len, emb_dim]

        outputs, hidden = self.lstm(embedded, hidden)
        # outputs: [batch_size, seq_len, hidden_dim]

        outputs = self.dropout(outputs)

        logits = self.fc_out(outputs)
        # logits: [batch_size, seq_len, vocab_size]

        return logits, hidden


VOCAB_SIZE = len(lm_vocab)
EMB_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3

lstm_lm = LSTMLanguageModel(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("LSTM language model initialized.")
print("Vocabulary size:", VOCAB_SIZE)
print("Trainable parameters:", count_parameters(lstm_lm))

LSTM language model initialized.
Vocabulary size: 16174
Trainable parameters: 16116014


In [11]:
# Forward pass test

x_batch, y_batch = next(iter(train_lm_loader))

x_batch = x_batch.to(device)

with torch.no_grad():
    logits, hidden = lstm_lm(x_batch)

print("Input batch shape:", x_batch.shape)
print("Logits shape:", logits.shape)
print("Expected logits shape: [batch_size, seq_len, vocab_size]")

Input batch shape: torch.Size([64, 50])
Logits shape: torch.Size([64, 50, 16174])
Expected logits shape: [batch_size, seq_len, vocab_size]


## 5.1 Training and Evaluating the LSTM

Training uses the **Adam optimizer** (lr = 0.001) with **gradient clipping** (max norm = 1.0) to stabilize recurrent training. The model is trained for up to **8 epochs**; the best checkpoint by validation loss is saved. **Perplexity** is computed as `exp(cross-entropy loss)` — a perplexity of *k* means the model is as uncertain as a uniform distribution over *k* words at each step; lower is better.

In [12]:
lm_optimizer = optim.Adam(lstm_lm.parameters(), lr=0.001)
lm_criterion = nn.CrossEntropyLoss()

def train_lm_epoch(model, dataloader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0.0

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, _ = model(x)

        vocab_size = logits.shape[-1]

        loss = criterion(
            logits.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


def evaluate_lm(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0.0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits, _ = model(x)

            vocab_size = logits.shape[-1]

            loss = criterion(
                logits.reshape(-1, vocab_size),
                y.reshape(-1)
            )

            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [13]:
# Train LSTM language model


N_EPOCHS = 8
CLIP = 1.0

best_valid_loss = float("inf")

lm_train_losses = []
lm_valid_losses = []

lm_training_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_lm_epoch(
        model=lstm_lm,
        dataloader=train_lm_loader,
        optimizer=lm_optimizer,
        criterion=lm_criterion,
        clip=CLIP
    )

    valid_loss = evaluate_lm(
        model=lstm_lm,
        dataloader=valid_lm_loader,
        criterion=lm_criterion
    )

    lm_train_losses.append(train_loss)
    lm_valid_losses.append(valid_loss)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(lstm_lm.state_dict(), "q5_best_lstm_lm.pt")

    epoch_time = time.time() - epoch_start

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Train PPL: {math.exp(train_loss):.2f} | "
        f"Valid PPL: {math.exp(valid_loss):.2f} | "
        f"Time: {epoch_time:.2f}s"
    )

lm_training_time = time.time() - lm_training_start

print("\nTraining completed.")
print("Best validation loss:", round(best_valid_loss, 4))
print("Best validation perplexity:", round(math.exp(best_valid_loss), 2))
print("Total training time in seconds:", round(lm_training_time, 2))
print("Model saved as q5_best_lstm_lm.pt")

Epoch 01 | Train Loss: 6.9335 | Valid Loss: 6.3333 | Train PPL: 1026.10 | Valid PPL: 563.02 | Time: 2.11s
Epoch 02 | Train Loss: 6.4759 | Valid Loss: 5.9837 | Train PPL: 649.32 | Valid PPL: 396.90 | Time: 1.72s
Epoch 03 | Train Loss: 6.1514 | Valid Loss: 5.7193 | Train PPL: 469.37 | Valid PPL: 304.69 | Time: 1.70s
Epoch 04 | Train Loss: 5.9123 | Valid Loss: 5.5688 | Train PPL: 369.57 | Valid PPL: 262.12 | Time: 1.71s
Epoch 05 | Train Loss: 5.7219 | Valid Loss: 5.4505 | Train PPL: 305.49 | Valid PPL: 232.87 | Time: 1.70s
Epoch 06 | Train Loss: 5.5725 | Valid Loss: 5.3809 | Train PPL: 263.09 | Valid PPL: 217.21 | Time: 1.70s
Epoch 07 | Train Loss: 5.4546 | Valid Loss: 5.3105 | Train PPL: 233.82 | Valid PPL: 202.45 | Time: 1.70s
Epoch 08 | Train Loss: 5.3531 | Valid Loss: 5.2674 | Train PPL: 211.25 | Valid PPL: 193.92 | Time: 1.70s

Training completed.
Best validation loss: 5.2674
Best validation perplexity: 193.92
Total training time in seconds: 14.06
Model saved as q5_best_lstm_lm.pt


## 5.2 Test Perplexity

The best LSTM checkpoint is loaded and evaluated on the test subset. Test perplexity is used as the main quantitative measure for the language model.

In [14]:
# Evaluate best LSTM model on test set

lstm_lm.load_state_dict(torch.load("q5_best_lstm_lm.pt", map_location=device))
lstm_lm.eval()

lstm_test_loss = evaluate_lm(
    model=lstm_lm,
    dataloader=test_lm_loader,
    criterion=lm_criterion
)

lstm_test_ppl = math.exp(lstm_test_loss)

print("LSTM test loss:", round(lstm_test_loss, 4))
print("LSTM test perplexity:", round(lstm_test_ppl, 2))

LSTM test loss: 5.1925
LSTM test perplexity: 179.91


## 6. Pretrained Transformer Language Model: GPT-2

**GPT-2** (Radford et al., 2019) is a 117M-parameter autoregressive Transformer decoder pre-trained on ~40GB of filtered web text (WebText). It uses **Byte Pair Encoding (BPE)** tokenization with a 50,257-token vocabulary, which handles rare words and morphological variation without an `<unk>` token.

GPT-2 is applied **without any fine-tuning** on WikiText-2. Perplexity is computed by sliding a window over each test text with a maximum context of **512 tokens**, passing each window through the model and averaging the per-token log-likelihoods. Because GPT-2 was trained on a vastly larger and more diverse corpus, its perplexity on WikiText-2 is expected to be substantially lower than the LSTM trained on the small subset.

In [15]:
TRANSFORMER_LM_NAME = "gpt2"

gpt2_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_LM_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(TRANSFORMER_LM_NAME)

# GPT-2 has no default padding token, so use EOS as padding if needed.
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

gpt2_model = gpt2_model.to(device)
gpt2_model.eval()

print("Transformer language model loaded successfully.")
print("Model:", TRANSFORMER_LM_NAME)
print("Device:", device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Transformer language model loaded successfully.
Model: gpt2
Device: cuda


In [16]:
# GPT-2 perplexity on WikiText-2 test subset


def compute_gpt2_perplexity(texts, tokenizer, model, max_length=512, max_examples=200):
    """
    Compute approximate perplexity for GPT-2 on a subset of text lines.
    Each non-empty line is evaluated independently.
    """
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    selected_texts = texts[:max_examples]

    with torch.no_grad():
        for text in selected_texts:
            if not text.strip():
                continue

            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length
            )

            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device)

            if input_ids.shape[1] < 2:
                continue

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=input_ids
            )

            # outputs.loss is the average negative log-likelihood per token.
            num_tokens = attention_mask.sum().item()

            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)

    return avg_loss, ppl


gpt2_test_loss, gpt2_test_ppl = compute_gpt2_perplexity(
    texts=test_texts,
    tokenizer=gpt2_tokenizer,
    model=gpt2_model,
    max_length=512,
    max_examples=200
)

print("GPT-2 test loss:", round(gpt2_test_loss, 4))
print("GPT-2 test perplexity:", round(gpt2_test_ppl, 2))

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


GPT-2 test loss: 3.8837
GPT-2 test perplexity: 48.6


## 7. Text Generation

Both models are prompted with the same three seed phrases to generate short continuations. The **LSTM** uses **temperature sampling** (temperature = 1.0) over the word-level vocabulary with a repetition control mechanism that prevents the same token from appearing more than twice consecutively. **GPT-2** uses **top-k / top-p (nucleus) sampling** (top-k = 50, top-p = 0.95, temperature = 0.8), which restricts sampling to the most probable tokens and produces more coherent, diverse output. The generated samples are compared qualitatively to highlight the representational capacity gap between the two models.

In [17]:
# LSTM text generation

def generate_lstm_text(
    model,
    prompt,
    vocab,
    itos,
    max_new_tokens=50,
    temperature=1.0
):
    """
    Generate text from the trained LSTM language model using temperature sampling.
    """

    model.eval()

    token_ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in simple_lm_tokenize(prompt)]

    if len(token_ids) == 0:
        token_ids = [vocab[EOS_TOKEN]]

    generated_ids = token_ids.copy()

    input_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(device)

    hidden = None

    with torch.no_grad():
        logits, hidden = model(input_tensor, hidden)

    current_token = input_tensor[:, -1]

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits, hidden = model(current_token.unsqueeze(1), hidden)

        next_token_logits = logits[:, -1, :] / temperature

        # Avoid generating padding.
        next_token_logits[:, PAD_IDX] = -1e10

        probabilities = torch.softmax(next_token_logits, dim=-1)

        next_token = torch.multinomial(probabilities, num_samples=1).item()

        generated_ids.append(next_token)

        if next_token == EOS_IDX:
            break

        current_token = torch.tensor([next_token], dtype=torch.long).to(device)

    return decode_lm_ids(generated_ids, itos)

In [18]:
# Step 9.1: GPT-2 text generation

def generate_gpt2_text(
    prompt,
    tokenizer,
    model,
    max_new_tokens=60,
    temperature=0.8,
    top_k=50,
    top_p=0.95
):
    """
    Generate text from GPT-2 using nucleus/top-k sampling.
    """

    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [19]:
# Generate samples from LSTM and GPT-2

prompts = [
    "The history of",
    "In the city",
    "The film was"
]

generated_samples = []

for prompt in prompts:
    lstm_sample = generate_lstm_text(
        model=lstm_lm,
        prompt=prompt,
        vocab=lm_vocab,
        itos=lm_itos,
        max_new_tokens=60,
        temperature=0.8
    )

    gpt2_sample = generate_gpt2_text(
        prompt=prompt,
        tokenizer=gpt2_tokenizer,
        model=gpt2_model,
        max_new_tokens=60,
        temperature=0.8,
        top_k=50,
        top_p=0.95
    )

    generated_samples.append({
        "prompt": prompt,
        "lstm_generation": lstm_sample,
        "gpt2_generation": gpt2_sample
    })

    print("=" * 100)
    print("PROMPT:")
    print(prompt)

    print("\nLSTM GENERATION:")
    print(lstm_sample)

    print("\nGPT-2 GENERATION:")
    print(gpt2_sample)

    print("\n")

PROMPT:
The history of

LSTM GENERATION:
the history of the end of a <unk> . <unk> makes the american <unk> language <unk> <unk> " and <unk> " in a <unk> , who had been able to the opportunity of the <unk> .

GPT-2 GENERATION:
The history of the United States is in many ways a tale of three separate empires. The first of these empires, the British Empire, came about through the efforts of King George III of England. This empire was formed following the death of his father, King George III, and the establishment of the United Kingdom of Great


PROMPT:
In the city

LSTM GENERATION:
in the city .

GPT-2 GENERATION:
In the city of New Brunswick, a police sergeant was shot to death after a suspected drug deal.

The alleged victim, who was an officer in the police force's narcotics unit, had been living in the town for several months.

On Tuesday, authorities said there were several "suspicious" arrests


PROMPT:
The film was

LSTM GENERATION:
the film was werneth with tellicherry ' s test 

## 8. Final Results and Export

Test perplexity for both models and the generated text samples are saved to CSV:

- **`q5_language_modeling_perplexity_results.csv`** — test loss and perplexity for LSTM and GPT-2
- **`q5_generated_text_samples.csv`** — prompt, LSTM output, and GPT-2 output for each seed phrase

In [20]:
q5_results_df = pd.DataFrame([
    {
        "model": "LSTM",
        "test_loss": lstm_test_loss,
        "test_perplexity": lstm_test_ppl
    },
    {
        "model": "GPT-2",
        "test_loss": gpt2_test_loss,
        "test_perplexity": gpt2_test_ppl
    }
])

q5_results_df_rounded = q5_results_df.copy()
q5_results_df_rounded["test_loss"] = q5_results_df_rounded["test_loss"].round(4)
q5_results_df_rounded["test_perplexity"] = q5_results_df_rounded["test_perplexity"].round(2)

q5_results_df_rounded

,model,test_loss,test_perplexity
0,LSTM,5.1925,179.91
1,GPT-2,3.8837,48.60


In [21]:
q5_samples_df = pd.DataFrame(generated_samples)

q5_results_df_rounded.to_csv("q5_language_modeling_perplexity_results.csv", index=False)
q5_samples_df.to_csv("q5_generated_text_samples.csv", index=False)

print("Saved files:")
print("- q5_language_modeling_perplexity_results.csv")
print("- q5_generated_text_samples.csv")

Saved files:
- q5_language_modeling_perplexity_results.csv
- q5_generated_text_samples.csv


In [22]:
print("Question 5 experiment completed.")
print("Dataset: WikiText-2")
print("Train lines:", len(train_texts))
print("Validation lines:", len(valid_texts))
print("Test lines:", len(test_texts))
print("Random seed:", SEED)
print("LSTM vocabulary size:", len(lm_vocab))
print("LSTM trainable parameters:", count_parameters(lstm_lm))
print("LSTM best validation loss:", round(best_valid_loss, 4))
print("LSTM best validation perplexity:", round(math.exp(best_valid_loss), 2))
print("LSTM test loss:", round(lstm_test_loss, 4))
print("LSTM test perplexity:", round(lstm_test_ppl, 2))
print("LSTM training time in seconds:", round(lm_training_time, 2))
print("Transformer model:", TRANSFORMER_LM_NAME)
print("GPT-2 test loss:", round(gpt2_test_loss, 4))
print("GPT-2 test perplexity:", round(gpt2_test_ppl, 2))

print("\nFinal perplexity results:")
display(q5_results_df_rounded)

Question 5 experiment completed.
Dataset: WikiText-2
Train lines: 8000
Validation lines: 1000
Test lines: 1000
Random seed: 42
LSTM vocabulary size: 16174
LSTM trainable parameters: 16116014
LSTM best validation loss: 5.2674
LSTM best validation perplexity: 193.92
LSTM test loss: 5.1925
LSTM test perplexity: 179.91
LSTM training time in seconds: 14.06
Transformer model: gpt2
GPT-2 test loss: 3.8837
GPT-2 test perplexity: 48.6

Final perplexity results:


,model,test_loss,test_perplexity
0,LSTM,5.1925,179.91
1,GPT-2,3.8837,48.60
